# 🎯 Real-Time Object Detection & Tracking — Interactive Demo

This notebook demonstrates the full pipeline:
1. **Detection** — YOLOv8 inference on images/video
2. **Feature Extraction** — SIFT/ORB keypoints
3. **Kalman Filter** — Bounding-box state prediction
4. **Tracking** — SORT & DeepSORT multi-object tracking
5. **MOT Metrics** — MOTA, MOTP, IDF1 evaluation
6. **Visualization** — Annotated frames with trails & HUD

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

print('Environment ready!')

## 1. Configuration & Setup

In [ ]:
from src.utils.config import load_config
from src.pipeline.pipeline_config import PipelineConfig

cfg = load_config('../configs/default.yaml')
pcfg = PipelineConfig.from_dict(cfg)

print(f'Detection backend:  {pcfg.backend}')
print(f'Tracker type:       {pcfg.tracker_type}')
print(f'Confidence thresh:  {pcfg.confidence_threshold}')
print(f'Input size:         {pcfg.input_size}')
print(f'Max age:            {pcfg.max_age}')
print(f'Min hits:           {pcfg.min_hits}')

## 2. Kalman Filter Demonstration

Visualize how the Kalman filter predicts and corrects bounding-box positions.

In [ ]:
from src.filters.kalman_filter import KalmanBoxFilter

# Simulate a box moving diagonally with noise
np.random.seed(42)
true_positions = []
measured_positions = []
predicted_positions = []
filtered_positions = []

kf = KalmanBoxFilter(np.array([100, 100, 150, 150], dtype=np.float64))

for t in range(50):
    # True position (constant velocity)
    true_x = 100 + t * 3
    true_y = 100 + t * 2
    true_box = np.array([true_x, true_y, true_x + 50, true_y + 50])
    true_positions.append((true_x + 25, true_y + 25))
    
    # Noisy measurement
    noise = np.random.randn(4) * 5
    measured = true_box + noise
    measured_positions.append(((measured[0] + measured[2]) / 2, (measured[1] + measured[3]) / 2))
    
    # Kalman predict
    pred = kf.predict()
    predicted_positions.append(((pred[0] + pred[2]) / 2, (pred[1] + pred[3]) / 2))
    
    # Kalman update
    upd = kf.update(measured)
    filtered_positions.append(((upd[0] + upd[2]) / 2, (upd[1] + upd[3]) / 2))

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
true_x, true_y = zip(*true_positions)
meas_x, meas_y = zip(*measured_positions)
filt_x, filt_y = zip(*filtered_positions)

ax.plot(true_x, true_y, 'g-', linewidth=2, label='Ground Truth')
ax.scatter(meas_x, meas_y, c='red', s=15, alpha=0.5, label='Noisy Measurements')
ax.plot(filt_x, filt_y, 'b-', linewidth=2, label='Kalman Filtered')

ax.set_xlabel('X position')
ax.set_ylabel('Y position')
ax.set_title('Kalman Filter — Tracking a Moving Object')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Feature Extraction — SIFT vs ORB

In [ ]:
from src.features.sift_extractor import SIFTExtractor
from src.features.orb_extractor import ORBExtractor

# Create a synthetic textured image
rng = np.random.RandomState(123)
img = rng.randint(0, 255, (480, 640, 3), dtype=np.uint8)
# Add some structure
cv2.rectangle(img, (100, 100), (300, 300), (0, 255, 0), 3)
cv2.circle(img, (450, 250), 80, (255, 0, 0), 3)
cv2.putText(img, 'DETECT', (200, 400), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)

# SIFT
sift = SIFTExtractor(max_keypoints=200)
sift_kps, sift_desc = sift.extract(img)
sift_img = cv2.drawKeypoints(img.copy(), sift_kps, None, color=(0, 255, 0))

# ORB
orb = ORBExtractor(max_keypoints=200)
orb_kps, orb_desc = orb.extract(img)
orb_img = cv2.drawKeypoints(img.copy(), orb_kps, None, color=(0, 0, 255))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(sift_img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'SIFT — {len(sift_kps)} keypoints ({sift.latency_ms:.1f} ms)')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(orb_img, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'ORB — {len(orb_kps)} keypoints ({orb.latency_ms:.1f} ms)')
axes[1].axis('off')

plt.suptitle('Feature Extraction Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. SORT Tracker Simulation

Simulate multi-object tracking with SORT on synthetic detections.

In [ ]:
from src.tracking.track import Track
from src.tracking.sort_tracker import SORTTracker

Track.reset_id_counter()
tracker = SORTTracker(max_age=10, min_hits=1, iou_threshold=0.3)

# Simulate 3 objects moving across frames
np.random.seed(42)
track_histories = {}
num_frames = 40

for t in range(num_frames):
    boxes = []
    # Object 1: moves right
    x1 = 50 + t * 5 + np.random.randn() * 2
    boxes.append([x1, 100, x1 + 60, 160])
    
    # Object 2: moves diagonally
    x2 = 200 + t * 3 + np.random.randn() * 2
    y2 = 200 + t * 2 + np.random.randn() * 2
    boxes.append([x2, y2, x2 + 50, y2 + 50])
    
    # Object 3: stationary
    boxes.append([400 + np.random.randn() * 2, 300 + np.random.randn() * 2, 470, 370])
    
    dets = {
        'boxes': np.array(boxes, dtype=np.float32),
        'scores': np.array([0.95, 0.88, 0.92], dtype=np.float32),
        'class_ids': np.array([0, 1, 0], dtype=np.int32),
    }
    
    tracks = tracker.update(dets)
    for trk in tracks:
        tid = trk.track_id
        cx = (trk.bbox[0] + trk.bbox[2]) / 2
        cy = (trk.bbox[1] + trk.bbox[3]) / 2
        if tid not in track_histories:
            track_histories[tid] = []
        track_histories[tid].append((cx, cy))

# Plot track trajectories
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for i, (tid, pts) in enumerate(track_histories.items()):
    xs, ys = zip(*pts)
    color = colors[i % len(colors)]
    ax.plot(xs, ys, '-o', color=color, markersize=3, label=f'Track {tid}')
    ax.annotate(f'ID {tid}', (xs[-1], ys[-1]), fontsize=10, fontweight='bold', color=color)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'SORT Tracker — {num_frames} Frames, {len(track_histories)} Tracks')
ax.legend()
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. MOT Metrics Evaluation

In [ ]:
from src.tracking.mot_metrics import evaluate_mot

# Build GT and predictions from our simulation
Track.reset_id_counter()
tracker2 = SORTTracker(max_age=10, min_hits=1, iou_threshold=0.3)

ground_truth = {}
predictions = {}

np.random.seed(42)
for t in range(30):
    # Ground truth: 2 objects
    gt_boxes = np.array([
        [50 + t * 5, 100, 110 + t * 5, 160],
        [200 + t * 3, 200 + t * 2, 250 + t * 3, 250 + t * 2],
    ], dtype=np.float64)
    gt_ids = np.array([1, 2])
    ground_truth[t] = {'ids': gt_ids, 'boxes': gt_boxes}
    
    # Noisy detections
    noise = np.random.randn(*gt_boxes.shape) * 3
    det_boxes = (gt_boxes + noise).astype(np.float32)
    
    dets = {
        'boxes': det_boxes,
        'scores': np.array([0.9, 0.85], dtype=np.float32),
        'class_ids': np.array([0, 1], dtype=np.int32),
    }
    tracks = tracker2.update(dets)
    
    if tracks:
        pred_ids = np.array([trk.track_id for trk in tracks])
        pred_boxes = np.array([trk.bbox for trk in tracks])
        predictions[t] = {'ids': pred_ids, 'boxes': pred_boxes}
    else:
        predictions[t] = {'ids': np.empty(0, dtype=int), 'boxes': np.empty((0, 4))}

metrics = evaluate_mot(ground_truth, predictions, iou_threshold=0.5)

print('\n📊 MOT Evaluation Results')
print('=' * 40)
for k, v in metrics.items():
    print(f'{k:>25s}: {v}')

## 6. Visualization Helpers Demo

In [ ]:
from src.utils.visualization import draw_detections, draw_hud

# Create a blank canvas
canvas = np.ones((480, 640, 3), dtype=np.uint8) * 40

# Draw some detections
boxes = np.array([
    [50, 80, 200, 280],
    [300, 150, 500, 400],
    [450, 50, 600, 180],
], dtype=np.float32)
scores = np.array([0.95, 0.87, 0.72], dtype=np.float32)
class_ids = np.array([0, 1, 2], dtype=np.int32)
class_names = ['person', 'car', 'bicycle']

draw_detections(canvas, boxes, scores, class_ids, class_names, thickness=2)
draw_hud(canvas, fps=28.5, num_tracks=3, latency_ms=18.2)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title('Visualization Output — Detections + HUD', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## Summary

| Component | Status |
|---|---|
| Configuration loading | ✅ |
| Kalman filter prediction & update | ✅ |
| SIFT / ORB feature extraction | ✅ |
| SORT multi-object tracking | ✅ |
| MOT metrics (MOTA, MOTP, IDF1) | ✅ |
| Visualization helpers | ✅ |

For full pipeline with YOLOv8 detection, run:
```bash
python scripts/run_pipeline.py --source 0         # webcam
python scripts/run_pipeline.py --source video.mp4  # video file
```